In [ ]:
# ============================================================
# 1. IMPORTS
# ============================================================

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

from packaging import version

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ============================================================
# 2. ENVIRONMENT CHECKS
# ============================================================

MIN_PYTHON = (3, 7)
MIN_SKLEARN = "1.0.1"

if sys.version_info < MIN_PYTHON:
    raise RuntimeError(
        f"Python {MIN_PYTHON[0]}.{MIN_PYTHON[1]}+ required."
    )

if version.parse(sklearn.__version__) < version.parse(MIN_SKLEARN):
    raise RuntimeError(
        f"Scikit-learn {MIN_SKLEARN}+ required."
    )

print("✓ Environment checks passed")

# ============================================================
# 3. LOAD DATA
# ============================================================

DATA_PATH = Path("btcusd_1-min_data.csv")

def load_bitcoin_data(path=DATA_PATH):
    if not path.exists():
        raise FileNotFoundError(
            f"Dataset not found: {path.resolve()}"
        )
    df = pd.read_csv(path)
    print(f"✓ Bitcoin data loaded ({df.shape[0]:,} rows)")
    return df

bitcoin = load_bitcoin_data()

# ============================================================
# 4. INITIAL DATA PREP
# ============================================================

bitcoin["Datetime"] = pd.to_datetime(
    bitcoin["Timestamp"],
    unit="s"
)

bitcoin = (
    bitcoin
    .set_index("Datetime")
    .drop(columns="Timestamp")
)

# ============================================================
# 5. RESAMPLE TO DAILY DATA
# ============================================================

df_daily = pd.DataFrame({
    "Open": bitcoin["Open"].resample("D").first(),
    "High": bitcoin["High"].resample("D").max(),
    "Low": bitcoin["Low"].resample("D").min(),
    "Close": bitcoin["Close"].resample("D").last(),
    "Volume": bitcoin["Volume"].resample("D").sum(),
})

df_daily.dropna(inplace=True)

print(
    f"✓ Resampled to {df_daily.shape[0]:,} daily observations"
)

# ============================================================
# 6. FEATURE ENGINEERING (Same as Classification)
# ============================================================

# Target for Regression: The actual price of the next close
df_daily["Next_Close"] = df_daily["Close"].shift(-1)

# Keep the exact same features
df_daily["Return_1d"] = df_daily["Close"].pct_change()
df_daily["MA_5"] = df_daily["Close"].rolling(5).mean()
df_daily["MA_20"] = df_daily["Close"].rolling(20).mean()
df_daily["High_Low_Spread"] = df_daily["High"] - df_daily["Low"]
df_daily["Dist_to_MA_5"] = (df_daily["Close"] - df_daily["MA_5"]) / df_daily["MA_5"]

# Drop NaNs introduced by rolling/shifting
df_daily.dropna(inplace=True)

feature_cols = [
    "Open",
    "High",
    "Low",
    "Close",
    "Volume",
    "Return_1d",
    "MA_5",
    "MA_20",
    "High_Low_Spread",
    "Dist_to_MA_5",
]

X = df_daily[feature_cols]
# Regression Target
y = df_daily["Next_Close"] 

print(f"✓ Training observations: {X.shape[0]}")

# ============================================================
# 7. CHRONOLOGICAL TRAIN / TEST SPLIT
# ============================================================

split_idx = int(len(X) * 0.8)

X_train = X.iloc[:split_idx]
X_test = X.iloc[split_idx:]

y_train = y.iloc[:split_idx]
y_test = y.iloc[split_idx:]

print(
    f"\nTrain: {X_train.shape[0]}"
    f"\nTest : {X_test.shape[0]}"
)

# ============================================================
# 8. MODELS
# ============================================================

lin_reg = LinearRegression()

pipe_lin = Pipeline([
    ("scaler", StandardScaler()),
    ("lin_reg", lin_reg),
])

# ============================================================
# 9. TRAIN FINAL MODEL & PREDICT
# ============================================================

# Fit Linear Regression on full train set
pipe_lin.fit(X_train, y_train)

# Predictions for model on the test set
y_pred_lin = pipe_lin.predict(X_test)

# ============================================================
# 10. EVALUATION METRICS
# ============================================================

print("\n📊 FINAL REPORT")
print("-" * 40)

def print_metrics(model_name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"{model_name}:")
    print(f"   MAE  : ${mae:,.2f}")
    print(f"   RMSE : ${rmse:,.2f}")
    print(f"   R²   :  {r2:.4f}\n")

print_metrics("Linear Regression", y_test, y_pred_lin)

# ============================================================
# 11. PLOT: PREDICTED VS ACTUAL (MACRO VIEW)
# ============================================================

plt.figure(figsize=(16, 7))

# Plot Actual Prices
plt.plot(
    y_test.index, 
    y_test.values, 
    label="Actual Next Close Price", 
    color="black", 
    linewidth=2,
    alpha=0.8
)

# Plot Linear Regression Predictions
plt.plot(
    y_test.index, 
    y_pred_lin, 
    label="Linear Regression", 
    color="royalblue", 
    linestyle="--",
    alpha=0.7
)

plt.title("Bitcoin Price Prediction: Actual vs. Predicted (Full Test Set)", fontsize=16)
plt.xlabel("Date", fontsize=12)
plt.ylabel("Price (USD)", fontsize=12)
plt.legend(loc="upper left", fontsize=12)
plt.grid(True, linestyle=":", alpha=0.6)
plt.tight_layout()

plt.show()

# ============================================================
# 12. PLOT: PREDICTED VS ACTUAL (ZOOMED IN TO EXPOSE LAG)
# ============================================================

# Select the last 60 days to clearly see the day-to-day lag
ZOOM_DAYS = 60

plt.figure(figsize=(16, 7))

# Plot Zoomed Actual Prices with markers
plt.plot(
    y_test.index[-ZOOM_DAYS:], 
    y_test.values[-ZOOM_DAYS:], 
    label="Actual Next Close Price", 
    color="black", 
    linewidth=2,
    marker='o', # Added marker
    alpha=0.8
)

# Plot Zoomed Predictions with markers
plt.plot(
    y_test.index[-ZOOM_DAYS:], 
    y_pred_lin[-ZOOM_DAYS:], 
    label="Linear Regression Prediction", 
    color="royalblue", 
    linestyle="--",
    marker='X', # Added marker
    markersize=8,
    alpha=0.7
)

plt.title(f"Zoomed In (Last {ZOOM_DAYS} Days): Exposing the 'Lag Effect'", fontsize=16)
plt.xlabel("Date", fontsize=12)
plt.ylabel("Price (USD)", fontsize=12)
plt.legend(loc="upper left", fontsize=12)
plt.grid(True, linestyle=":", alpha=0.6)
plt.tight_layout()

plt.show()